In [3]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs

# Configuración de conexión a MinIO en el VPS
MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"      # Reemplaza con tu Access Key real
MINIO_SECRET = "adminGrupo3"  # Reemplaza con tu Secret Key real
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

# Sistema de archivos S3 para escritura directa en un único Parquet por dataset
fs = s3fs.S3FileSystem(
    key=MINIO_KEY, 
    secret=MINIO_SECRET, 
    client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)

CHUNK_SIZE = 100000

# ==============================================================================
# 1. LIMPIEZA DE TRAIN_TRANSACTION.CSV (POR CHUNKS -> 1 ARCHIVO PARQUET)
# ==============================================================================
print("🧹 1. Iniciando limpieza de train_transaction.csv mediante Chunking...")

ruta_trans_bronze = f"s3://{BUCKET_NAME}/bronze/train_transaction.csv"
ruta_trans_silver = f"{BUCKET_NAME}/silver/train_transaction_clean.parquet"

# Seleccionamos las columnas principales para estandarizar el esquema
cols_trans = [
    'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
    'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'dist1', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'D1', 'M4'
]

writer_trans = None
total_filas_trans = 0
duplicados_trans = 0
inconsistencias_trans = 0

with fs.open(ruta_trans_silver, 'wb') as f_out:
    for i, chunk in enumerate(pd.read_csv(
        ruta_trans_bronze, 
        usecols=cols_trans, 
        chunksize=CHUNK_SIZE, 
        storage_options=storage_opts
    )):
        # A. Duplicados
        dup = chunk.duplicated(subset=['TransactionID']).sum()
        duplicados_trans += dup
        chunk = chunk.drop_duplicates(subset=['TransactionID']).copy()
        
        # B. Inconsistencias (montos menores o iguales a 0 y categorías mal escritas)
        inc = (chunk['TransactionAmt'] <= 0).sum()
        inconsistencias_trans += inc
        chunk = chunk[chunk['TransactionAmt'] > 0]
        chunk['card6'] = chunk['card6'].replace({'debit or credit': 'debit', 'charge card': 'credit'})
        
        # C. Valores faltantes (Imputación categórica y numérica)
        for col_cat in ['card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4', 'ProductCD']:
            chunk[col_cat] = chunk[col_cat].fillna('desconocido').astype(str)
            
        for col_num in ['card1', 'card2', 'card3', 'card5', 'addr1', 'dist1', 'C1', 'C2', 'D1']:
            mediana = chunk[col_num].median()
            chunk[col_num] = chunk[col_num].fillna(mediana if pd.notna(mediana) else 0.0).astype('float32')
            
        # D. Corrección de tipos de datos
        chunk['TransactionID'] = chunk['TransactionID'].astype('int32')
        chunk['isFraud'] = chunk['isFraud'].astype('int8')
        chunk['TransactionDT'] = chunk['TransactionDT'].astype('int32')
        chunk['TransactionAmt'] = chunk['TransactionAmt'].astype('float32')
        
        # E. Escribir el chunk dentro del único archivo Parquet de transacciones
        tabla_pa = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer_trans is None:
            writer_trans = pq.ParquetWriter(f_out, tabla_pa.schema, compression='snappy')
        writer_trans.write_table(tabla_pa)
        
        total_filas_trans += len(chunk)
        print(f"   • Bloque Transacciones #{i+1} procesado ({len(chunk):,} filas)")

    if writer_trans:
        writer_trans.close()

print(f"✅ train_transaction_clean.parquet guardado en Silver!")
print(f"   [Validación] Filas limpias: {total_filas_trans:,} | Duplicados: {duplicados_trans} | Inconsistencias: {inconsistencias_trans}\n")


# ==============================================================================
# 2. LIMPIEZA DE TRAIN_IDENTITY.CSV (POR CHUNKS -> 1 ARCHIVO PARQUET)
# ==============================================================================
print("🧹 2. Iniciando limpieza de train_identity.csv mediante Chunking...")

ruta_id_bronze = f"s3://{BUCKET_NAME}/bronze/train_identity.csv"
ruta_id_silver = f"{BUCKET_NAME}/silver/train_identity_clean.parquet"

cols_id = ['TransactionID', 'id_01', 'id_02', 'id_12', 'id_15', 'id_30', 'id_31', 'DeviceType', 'DeviceInfo']

writer_id = None
total_filas_id = 0
duplicados_id = 0

with fs.open(ruta_id_silver, 'wb') as f_out_id:
    for i, chunk_id in enumerate(pd.read_csv(
        ruta_id_bronze, 
        usecols=cols_id, 
        chunksize=CHUNK_SIZE, 
        storage_options=storage_opts
    )):
        # A. Duplicados
        dup_id = chunk_id.duplicated(subset=['TransactionID']).sum()
        duplicados_id += dup_id
        chunk_id = chunk_id.drop_duplicates(subset=['TransactionID']).copy()
        
        # B. Inconsistencias (estandarizar texto en minúsculas/sin espacios extra)
        chunk_id['DeviceType'] = chunk_id['DeviceType'].astype(str).str.lower().str.strip()
        chunk_id['DeviceType'] = chunk_id['DeviceType'].replace({'nan': 'desconocido', 'none': 'desconocido'})
        
        # C. Valores faltantes
        for col_cat in ['id_12', 'id_15', 'id_30', 'id_31', 'DeviceInfo']:
            chunk_id[col_cat] = chunk_id[col_cat].fillna('desconocido').astype(str)
            
        for col_num in ['id_01', 'id_02']:
            mediana_id = chunk_id[col_num].median()
            chunk_id[col_num] = chunk_id[col_num].fillna(mediana_id if pd.notna(mediana_id) else 0.0).astype('float32')
            
        # D. Corrección de tipos de datos
        chunk_id['TransactionID'] = chunk_id['TransactionID'].astype('int32')
        
        # E. Escribir el chunk dentro del único archivo Parquet de identidad
        tabla_pa_id = pa.Table.from_pandas(chunk_id, preserve_index=False)
        if writer_id is None:
            writer_id = pq.ParquetWriter(f_out_id, tabla_pa_id.schema, compression='snappy')
        writer_id.write_table(tabla_pa_id)
        
        total_filas_id += len(chunk_id)
        print(f"   • Bloque Identidad #{i+1} procesado ({len(chunk_id):,} filas)")

    if writer_id:
        writer_id.close()

print(f"✅ train_identity_clean.parquet guardado en Silver!")
print(f"   [Validación] Filas limpias: {total_filas_id:,} | Duplicados: {duplicados_id}")

🧹 1. Iniciando limpieza de train_transaction.csv mediante Chunking...
   • Bloque Transacciones #1 procesado (100,000 filas)
   • Bloque Transacciones #2 procesado (100,000 filas)
   • Bloque Transacciones #3 procesado (100,000 filas)
   • Bloque Transacciones #4 procesado (100,000 filas)
   • Bloque Transacciones #5 procesado (100,000 filas)
   • Bloque Transacciones #6 procesado (90,540 filas)
✅ train_transaction_clean.parquet guardado en Silver!
   [Validación] Filas limpias: 590,540 | Duplicados: 0 | Inconsistencias: 0

🧹 2. Iniciando limpieza de train_identity.csv mediante Chunking...
   • Bloque Identidad #1 procesado (100,000 filas)
   • Bloque Identidad #2 procesado (44,233 filas)
✅ train_identity_clean.parquet guardado en Silver!
   [Validación] Filas limpias: 144,233 | Duplicados: 0


In [4]:
# Leer metadatos y muestra de los nuevos archivos Parquet en Silver
df_t_silver = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_transaction_clean.parquet", storage_options=storage_opts)
df_i_silver = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_identity_clean.parquet", storage_options=storage_opts)

print("=== RESUMEN POST-LIMPIEZA (CAPA SILVER) ===")
print(f"Transacciones Silver: {df_t_silver.shape} | Total Nulos: {df_t_silver.isnull().sum().sum()}")
print(f"Identidades Silver:   {df_i_silver.shape} | Total Nulos: {df_i_silver.isnull().sum().sum()}\n")

print("Tipos de datos optimizados en Transacciones:")
print(df_t_silver.dtypes.value_counts())

=== RESUMEN POST-LIMPIEZA (CAPA SILVER) ===
Transacciones Silver: (590540, 19) | Total Nulos: 0
Identidades Silver:   (144233, 9) | Total Nulos: 3423

Tipos de datos optimizados en Transacciones:
float32    10
str         6
int32       2
int8        1
Name: count, dtype: int64


In [5]:
# 1. Rellenar los 3,423 nulos restantes en DeviceType y asegurar todas las columnas de identidad
df_i_silver['DeviceType'] = df_i_silver['DeviceType'].fillna('desconocido')
df_i_silver = df_i_silver.fillna('desconocido')

# 2. Sobrescribir únicamente el archivo de identidad en la Capa Silver de MinIO
df_i_silver.to_parquet(
    f"s3://{BUCKET_NAME}/silver/train_identity_clean.parquet", 
    index=False, 
    storage_options=storage_opts
)

# 3. Verificar que ambos datasets quedaron con 0 nulos
print("=== VERIFICACIÓN ACTUALIZADA (CAPA SILVER) ===")
print(f"Transacciones Silver: {df_t_silver.shape} | Total Nulos: {df_t_silver.isnull().sum().sum()}")
print(f"Identidades Silver:   {df_i_silver.shape} | Total Nulos: {df_i_silver.isnull().sum().sum()}")

=== VERIFICACIÓN ACTUALIZADA (CAPA SILVER) ===
Transacciones Silver: (590540, 19) | Total Nulos: 0
Identidades Silver:   (144233, 9) | Total Nulos: 0
